# FINM 32000: Homework 4

By Andrew McLaughlin

## Problem 1

Assume that the short rate (the instantaneous spot rate of interest) follows the process

$$
dr_t = \mu(r_t, t)\,dt + \sigma(r_t, t)\,dW_t
$$

where $W_t$ is Brownian motion under risk-neutral probabilities. This framework includes models such as the Vasicek and CIR models, which correspond to particular choices of the functions $(\mu, \sigma)$, but for part (a), let's leave $\mu$ and $\sigma$ as unspecified functions.

### (a)

Consider an interest rate derivative whose time-$T$ payout has value given by some function $F(r_T)$, and whose time-$t$ price $C_t$ satisfies $C_t = C(r_t, t)$ for some smooth pricing function $C$.

Apply Ito's rule to find the risk-neutral dynamics of $C$. Then set its drift equal to $rC$, to derive a PDE for $C(r, t)$.

---

Suppose, in particular, that the risk-neutral dynamics of $r$ are given by a Vasicek model

$$
dr_t = \kappa(\theta - r_t)\,dt + \sigma\,dW_t,
$$

with parameters $\kappa = 3$, $\theta = 0.05$, $\sigma = 0.03$. Consider a $T = 5$-year discount bond (a zero-coupon bond which pays 1 at maturity $T$).

Suppose, in particular, that the risk-neutral dynamics of $r$ are given by a Vasicek model

$$
dr_t = \kappa(\theta - r_t)\,dt + \sigma\,dW_t,
$$

with parameters $\kappa = 3$, $\theta = 0.05$, $\sigma = 0.03$. Consider a $T = 5$-year discount bond (a zero-coupon bond which pays 1 at maturity $T$).

### (b)

Write code to find the time-0 price of the bond by applying a standard central-difference explicit finite difference scheme to the PDE in (a). (Therefore $C_j^n$ will be determined by $C_{j+1}^{n+1}$, $C_j^{n+1}$, and $C_{j-1}^{n+1}$.)

Complete the code in the file `finm320-26-hw4.ipynb`.

In [45]:
import numpy as np

In [46]:
class Vasicek:

    def __init__(self,kappa,theta,sigma):
        self.kappa=kappa
        self.theta=theta
        self.sigma=sigma

In [47]:
hw41dynamics = Vasicek(kappa=3,theta=0.05,sigma=0.03)

In [48]:
class Bond:

    def __init__(self, T):
        self.T=T


In [49]:
hw41contract = Bond(T=5)

In [50]:
class FDexplicitEngine:

    def __init__(self, rMax, rMin, deltar, deltat, useUpwind):
        self.rMax=rMax
        self.rMin=rMin
        self.deltar=deltar
        self.deltat=deltat
        self.useUpwind=useUpwind

    def price_bond_vasicek(self,contract,dynamics):
    # You complete the coding of this function
    #
    # Returns array of all initial short rates,
    # and the corresponding array of zero-coupon
    # T-maturity bond prices

        T = contract.T
        N=round(T/self.deltat)
        if abs(N-T/self.deltat) > 1e-12:
            raise ValueError("Bad delta t")

        r=np.arange(self.rMax,self.rMin-self.deltar/2,-self.deltar)   #I'm making the FIRST indices of the array correspond to HIGH levels of r
        bondprice=np.ones(np.size(r))


        if self.useUpwind:
            mu  = dynamics.kappa * (dynamics.theta - r)
            a   = dynamics.sigma**2 * self.deltat / self.deltar**2
            qu  = 0.5 * a + np.maximum(mu, 0) * self.deltat / self.deltar
            qd  = 0.5 * a + np.maximum(-mu, 0) * self.deltat / self.deltar
            qm  = 1 - a - np.abs(mu) * self.deltat / self.deltar
        else:
            qu = 0.5 * (dynamics.sigma**2 * self.deltat / self.deltar**2 + dynamics.kappa * (dynamics.theta - r) * self.deltat / self.deltar)
            qd = 0.5 * (dynamics.sigma**2 * self.deltat / self.deltar**2 - dynamics.kappa * (dynamics.theta - r) * self.deltat / self.deltar)
            qm = (1 - dynamics.sigma**2 * self.deltat / self.deltar**2) * np.ones(np.size(r))



        for t in np.arange(N-1,-1,-1)*self.deltat:
            # Do not change any of the code in this loop

            bondprice[1:-1]=1/(1+r[1:-1]*self.deltat)*(qd[1:-1]*bondprice[2:]+qm[1:-1]*bondprice[1:-1]+qu[1:-1]*bondprice[:-2])
            # We are only calculating the interior grid points here, so
            # bondprice, r, qd, qm, and qu have [1:-1] indexes

            # For this contract, it is not obvious
            # what boundary conditions to use at the top and bottom,
            # so let us assume "linearity" boundary conditions
            bondprice[0]=2*bondprice[1]-bondprice[2]
            bondprice[-1]=2*bondprice[-2]-bondprice[-3]

        return (r, bondprice)

In [51]:
hw41FD = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=False)

In [52]:
(r, bondprice) = hw41FD.price_bond_vasicek(hw41contract,hw41dynamics)

In [53]:
np.set_printoptions(precision=4,suppress=True)
displayrows=(r<0.15+hw41FD.deltar/2) & (r>0.0-hw41FD.deltar/2)

In [54]:
print(np.stack((r, bondprice),axis=1)[displayrows])

[[ 1.5000e-01 -1.4273e+09]
 [ 1.4000e-01  1.6361e+08]
 [ 1.3000e-01  2.2294e+07]
 [ 1.2000e-01 -1.3724e+06]
 [ 1.1000e-01 -1.3361e+05]
 [ 1.0000e-01  3.2966e+03]
 [ 9.0000e-02  1.3021e+02]
 [ 8.0000e-02  7.7128e-01]
 [ 7.0000e-02  7.7385e-01]
 [ 6.0000e-02  7.7643e-01]
 [ 5.0000e-02  7.7902e-01]
 [ 4.0000e-02  7.8162e-01]
 [ 3.0000e-02  7.8423e-01]
 [ 2.0000e-02  7.8685e-01]
 [ 1.0000e-02  1.4165e+03]
 [-3.3307e-16  5.1498e+04]]


### (c)

Also write code to price the bond using an explicit upwind approximation to $\frac{\partial C}{\partial r}$ instead of the usual central difference. Specifically, for those $r_j$ such that $\kappa(\theta - r_j) \geq 0$, approximate $\frac{\partial C}{\partial r}(r_j, t_{n+1})$ using the points $C_{j+1}^{n+1}$ and $C_j^{n+1}$. For those $r_j$ such that $\kappa(\theta - r_j) < 0$, approximate $\frac{\partial C}{\partial r}(r_j, t_{n+1})$ using the points $C_j^{n+1}$ and $C_{j-1}^{n+1}$. (For $\frac{\partial^2 C}{\partial r^2}$, use the usual central-difference approximation.)

In (b) and (c), to approximate the PDE's $rC$ term, use the values of $r$ and $C$ at node $(n, j)$. (As we said in class, node $(n+1, j)$ would also be a natural choice, but let's choose $n$ instead of $n+1$.) At the grid's upper and lower boundaries $r_{\max}$ and $r_{\min}$, impose for all $t < T$ the "linearity" boundary conditions:

$$
C(r_{\max}, t) = 2C(r_{\max} - \Delta r, t) - C(r_{\max} - 2\Delta r, t)
$$

$$
C(r_{\min}, t) = 2C(r_{\min} + \Delta r, t) - C(r_{\min} + 2\Delta r, t)
$$

(This technique can help in some situations where it is not obvious what boundary conditions to use.) Thus, in each column of the grid, first solve for $C$ in the interior nodes; then deal with the top and bottom nodes.

In [55]:
hw41FD = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=True)
(r, bondprice) = hw41FD.price_bond_vasicek(hw41contract,hw41dynamics)
np.set_printoptions(precision=4,suppress=True)
displayrows=(r<0.15+hw41FD.deltar/2) & (r>0.0-hw41FD.deltar/2)
print(np.stack((r, bondprice),axis=1)[displayrows])

[[ 0.15    0.7536]
 [ 0.14    0.7561]
 [ 0.13    0.7586]
 [ 0.12    0.7611]
 [ 0.11    0.7637]
 [ 0.1     0.7662]
 [ 0.09    0.7688]
 [ 0.08    0.7713]
 [ 0.07    0.7739]
 [ 0.06    0.7765]
 [ 0.05    0.7791]
 [ 0.04    0.7817]
 [ 0.03    0.7843]
 [ 0.02    0.7869]
 [ 0.01    0.7895]
 [-0.      0.7922]]


### (d)

Suppose $f : \mathbb{R} \to \mathbb{R}$ is smooth in some open neighborhood of $x$. Show that as $h \to 0$,

$$
\frac{f(x+h) - f(x)}{h} - f'(x) = O(h)
\qquad \text{and} \qquad
\frac{f(x+h) - f(x-h)}{2h} - f'(x) = O(h^2)
$$

using Taylor's theorem. The $O(h)$ means "some function bounded by a constant times $h$, near $h = 0$." Likewise, $O(h^2)$ means "some function bounded by a constant times $h^2$, near $h = 0$." Different instances of "$O$" may mean different functions. The "constants" may depend on $x$ but not $h$.


### (e)

For all part (e) calculations: Use the grid spacings $\Delta r = 0.01$ and $\Delta t = 0.01$. Use $r_{\max} = 0.35$ and $r_{\min} = -0.25$ for the upper and lower boundaries of the grid, respectively.

Run a central-difference calculation and an upwind calculation of the bond price for $r_0 = 0.10$. Which is more accurate? The more accurate of the two solutions should agree, to three significant digits, with the exact bond price in this model: **0.7661**. The less accurate of the two solutions will be very inaccurate.

**Answer:** The upwind method is more accurate.

### (f)

Based on your answers to (d) and (e), insert either **"greater"** or **"less"** in each blank space in the following rule-of-thumb. No explanation necessary.

> Ignoring stability issues and considering only consistency (i.e. "truncation error," also known as "local discretization error"), the upwind explicit scheme, which uses one-sided spatial differences, discretizes the PDE with ________ accuracy than the standard explicit scheme, which uses central spatial differences.
>
> However, to actually guarantee convergence, the grid spacing must satisfy certain stability constraints, to prevent errors from propagating explosively. In a PDE exhibiting strong drift, we have seen that these constraints may allow the upwind scheme ________ freedom in choosing grid spacing, compared to the central scheme.

**Answer:** less, greater


### (g)

The continuously-compounded yield-to-maturity of a zero-coupon bond with time-$t$ price $P_t$ and nonrandom face value $P_T$ to be paid at maturity date $T$ is

$$
\frac{\log(P_T / P_t)}{T - t}
$$

where, as always for us, $\log$ denotes natural log, and where $P_T = 1$ according to this problem's assumptions. One way to think of the time-$t$ yield to maturity $T$ is as the average of some type of time-$t$ expectation of the instantaneous spot rates from time $t$ to time $T$.

Find the yield-to-maturity of a 5-year discount bond, in the case that $r_0 = 0.12$, and in the case that $r_0 = 0.02$. (The "good" results from part (e) may be used here. The "bad" results should not be used, unless you want to fix them by modifying the grid spacings.)

Why, intuitively, is the yield for $r_0 = 0.12$ smaller than $0.12$, whereas the yield for $r_0 = 0.02$ is greater than $0.02$?

> **Comment:** Under these short-rate dynamics, there do exist analytic pricing formulas for bonds. So we do not need finite difference methods to value the simple payoff that we have here. But the finite difference scheme can be modified to handle contracts for which exact pricing formulas do not exist.


In [56]:
ytm = -np.log(bondprice) / hw41contract.T
print(np.stack((r, bondprice, ytm), axis=1)[displayrows])


[[ 0.15    0.7536  0.0566]
 [ 0.14    0.7561  0.0559]
 [ 0.13    0.7586  0.0553]
 [ 0.12    0.7611  0.0546]
 [ 0.11    0.7637  0.0539]
 [ 0.1     0.7662  0.0533]
 [ 0.09    0.7688  0.0526]
 [ 0.08    0.7713  0.0519]
 [ 0.07    0.7739  0.0513]
 [ 0.06    0.7765  0.0506]
 [ 0.05    0.7791  0.0499]
 [ 0.04    0.7817  0.0493]
 [ 0.03    0.7843  0.0486]
 [ 0.02    0.7869  0.0479]
 [ 0.01    0.7895  0.0473]
 [-0.      0.7922  0.0466]]


**Answer:** Under the Vasicek mdoel, the yield to maturity (YTM) will revert to $\theta = 0.05$. If $\theta$ is the long run mean of the short rate for the bond, then $r_t$ will move towards $0.05$ over time. If $r_0$ is$0.12$, the the YTM is expected to fall towards $0.05$, so the averaage rate over 5 years is expected to be less than $0.12$. If $r_0$ is $0.02$, the the YTM is expected to rise towards $0.05$, so the averaage rate over 5 years is expected to be greater than $0.02$. 

---

## Problem 2

The interest rate on the bank account is $r$. For $0 \leq t \leq T$, let $X_t$ be a time-$t$ futures or forward price, with expiration date $T$, on some underlying. (Futures prices = forward prices, if the interest rate is nonrandom, as it is here.) Under risk-neutral probabilities, assume $X$ has CEV dynamics

$$
dX_t = \sigma X_t^{1+\alpha}\,dW_t, \qquad X_0 = 100
$$

with constants $\sigma$, $\alpha$. The superscript on $X_t$ is an exponent (power). As a futures/forward price, $X$ has drift coefficient 0 (but prices of European options on $X$ still have drift coefficient $r$).

### (a)

Let $C(X_t, t)$ be the time-$t$ no-arbitrage price of a European put on $X$, with strike $K$ and expiry $T$. Write down a PDE, with terminal condition, for $C(X, t)$. Leave your answer in terms of $r$, $\nu$, $\alpha$, $K$, $T$.

### (b)

Let $r = 0.05$, $\sigma = 3$, $\alpha = -0.5$. Use Crank-Nicolson to find the time-0 price of an American put on $X$ with strike $K = 100$ and expiry $T = 0.25$. Partial code is provided in the `ipynb` file. You may use the boundary conditions implemented in the function `FD_CrankNicolson_Engine.price_put_CEV`.

At the low-$X$ boundary, it assumes the put value equals intrinsic value (exercise value). At the high-$X$ boundary, it approximates the put value as zero. You may use the FD grid given in the `ipynb` file.

In [57]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [58]:
class CEV:

    def __init__(self,volcoeff,alpha,rGrow,r,X0):
        self.volcoeff = volcoeff
        self.alpha = alpha
        self.rGrow = rGrow
        self.r = r
        self.X0 = X0


In [59]:
hw42dynamics = CEV(volcoeff=3, alpha=-0.5, rGrow=0, r=0.05, X0=100)

In [60]:
class Put:

    def __init__(self,T,K):
        self.T = T;
        self.K = K;

In [61]:
hw42contract=Put(T=0.25,K=100)

In [62]:
class FD_CrankNicolson_Engine:

    def __init__(self,XMax,XMin,deltaX,deltat):
        self.XMax=XMax
        self.XMin=XMin
        self.deltaX=deltaX
        self.deltat=deltat

    #You complete the coding of this function:
    def TicksAndMatricesCEV(self,T,dynamics):

        alpha, r, rGrow, volcoeff = dynamics.alpha, dynamics.r, dynamics.rGrow, dynamics.volcoeff

        N=round(T/self.deltat)
        if abs(N-T/self.deltat)>1e-12:
            raise ValueError('Bad time step')
        numX=round((self.XMax-self.XMin)/self.deltaX)+1
        if abs(numX-(self.XMax-self.XMin)/self.deltaX-1)>1e-12:
            raise ValueError('Bad time step')
        X=np.linspace(self.XMax,self.XMin,numX)    #The FIRST indices in this array are for HIGH levels of X
        tTicks = np.arange(N-1,-1,-1)*self.deltat

        ratio1 = self.deltat/self.deltaX
        ratio2 = self.deltat/self.deltaX**2

        f = 0.5 * volcoeff**2 * X**(2*(1+alpha))
        g = rGrow * X
        h = -r


        F = 0.5*ratio2*f + 0.25*ratio1*g
        G =     ratio2*f - 0.50*self.deltat*h
        H = 0.5*ratio2*f - 0.25*ratio1*g

        #Right-hand-side matrix
        RHSmatrix = diags([H[:-1], 1-G, F[1:]], [1,0,-1], shape=(numX,numX), format="csr")

        #Left-hand-side matrix
        LHSmatrix = diags([-H[:-1], 1+G, -F[1:]], [1,0,-1], shape=(numX,numX), format="csr")
        # diags creates SPARSE matrices

        return(X, tTicks, LHSmatrix, RHSmatrix, H[-1], F[0])


    #You complete the coding of this function:
    def price_put_CEV(self,contract,dynamics):

        # returns array of all initial X levels,
        # and the corresponding array of put prices

        X, tTicks, LHSmatrix, RHSmatrix, bottomH, topF = self.TicksAndMatricesCEV(contract.T,dynamics)
        # The X array contains the _interior_ levels of the grid,
        # from the smallest XMin to the largest XMax
        # The boundary conditions are imposed one level _beyond_,
        # e.g. at X_lowboundary=XMin-deltaX, not at XMin.
        # To relate to lecture notation, X_lowboundary is X_{-J}
        # whereas XMin is X_{-J+1}

        putprice=np.maximum(contract.K-X,0)
        X_lowboundary=self.XMin-self.deltaX

        for t in tTicks:

            rhs = RHSmatrix @ putprice

            #Now let's add the boundary condition vectors.
            #They are nonzero only in the last component:
            rhs[-1]=rhs[-1]+2*bottomH*(contract.K-X_lowboundary)

            putprice = spsolve(LHSmatrix, rhs)                  #You code this.

            # Three possibilities are:
            # 1. scipy.linalg.solve
            # 2. scipy.sparse.linalg.spsolve
            # 3. scipy.linalg.solve_banded
            #
            # 1. is not recommended here, because it does not take advantage of the fact that our matrix is mostly zeros.
            # 2. is more efficient (faster, and uses less storage) in this case, by recognizing our matrix's sparse structure.
            # 3. is the most efficient, by recognizing the banded (specifically, tridiagonal) matrix structure in our case.
            #
            # For convenience, I chose to set up RHSmatrix and LHSmatrix for approach 2 (spsolve)
            # because 3 (solve_banded) doesn't implement a banded matrix "class"
            # that supports matrix multiplication seamlessly (which approach 2 does)

            putprice = np.maximum(putprice, contract.K-X)

        return(X, putprice)

In [63]:
hw42FD = FD_CrankNicolson_Engine(XMax=200,XMin=50,deltaX=0.1,deltat=0.0005)

In [64]:
(X0_all, putprice) = hw42FD.price_put_CEV(hw42contract,hw42dynamics)

In [65]:
# pricer_put_CEV_CrankNicolson gives us option prices for ALL X0 from XMin to XMax
# But let's display only for a few X0 near 100:

displayStart = hw42dynamics.X0-hw42FD.deltaX*1.5
displayEnd   = hw42dynamics.X0+hw42FD.deltaX*1.5
displayrows  = (X0_all>displayStart) & (X0_all<displayEnd)
np.set_printoptions(precision=4, suppress=True)
print(np.stack((X0_all, putprice),axis=1)[displayrows])

[[100.1      5.8704]
 [100.       5.9183]
 [ 99.9      5.9665]]


### (c)

Compute numerically the time-0 delta and gamma of the put in (b).

In [66]:
idx = np.argmin(np.abs(X0_all - 100))
dX = X0_all[idx+1] - X0_all[idx-1]

delta = (putprice[idx+1] - putprice[idx-1]) / dX
gamma = (putprice[idx+1] - 2*putprice[idx] + putprice[idx-1]) / (dX/2)**2

print(f'Delta: {delta:.6f}')
print(f'Gamma: {gamma:.6f}')


Delta: -0.480640
Gamma: 0.026400


### (d)

Using exactly the same `FD_CrankNicolson_Engine.price_put_CEV` function as in (b) — meaning that you can change the input passed into the function, but cannot change the function's code — find the time-0 price of the American put in (b), but assuming Black-Scholes dynamics for $X$ with volatility $0.30$ and interest rate $0.05$ and $X_0 = 100$.

In [70]:
hw4dynamics_BS = CEV(volcoeff=0.30, alpha=0, rGrow=0, r=0.05, X0=100)
(X0_all_BS, putprice_BS) = hw42FD.price_put_CEV(hw42contract, hw4dynamics_BS)

idx_BS = np.argmin(np.abs(X0_all_BS - 100))
print(f'American put price: {putprice_BS[idx_BS]:.4f}')


American put price: 5.9169


### (e)

Intuitively, how does the shape of the European options-implied volatility skew (as a function of strike) differ, between the (b) dynamics vs. the (d) dynamics? If you wish, you may actually compute European implied volatilities, but this is not required; the intuition is enough here.

**Answer:** CEV dynamics from part(b) introduce a downward-sloping implied vol. With $\alpha = 0.5$ then local vol $= 3X^{-0.5}$ which is a decreasing function. Whereas, Black-Scholes assumes constant volatitlity regardless of strike; hence, the skew would be flat.